# Colab 一键运行：YOLOv8n/YOLOv8m 默认与调整参数消融实验

本 Notebook 用于在 Google Colab 中一次性运行 4 组实验，并自动生成结果表、曲线图和压缩包。

- E1: `yolov8n.pt` + 默认检测参数：`imgsz=640, conf=0.25, iou=0.70`
- E2: `yolov8n.pt` + 高召回调整参数：`imgsz=1280, conf=0.05, iou=0.85`
- E3: `yolov8m.pt` + 默认检测参数：`imgsz=640, conf=0.25, iou=0.70`
- E4: `yolov8m.pt` + 高召回调整参数：`imgsz=1280, conf=0.05, iou=0.85`

使用方法：先在 Colab 菜单选择 `代码执行程序 -> 更改运行时类型 -> T4 GPU`，然后从上到下运行全部单元格。若要跑完整视频，把下面配置单元格中的 `MAX_FRAMES = 0`；若只想快速验证，保留 `MAX_FRAMES = 300`。

In [4]:
# 1. 基础配置
# 如果你的项目已经在 /content/YOLOv8-DeepSORT-Object-Tracking 或当前目录，这里无需修改。
# 如果需要自动从 Git 仓库克隆项目，把 REPO_URL 改成你的仓库地址。

from pathlib import Path
import csv
import os
import shutil
import subprocess
import sys
import urllib.request

REPO_URL = "https://github.com/ZiDXie/YOLOv8-DeepSORT-Object-Tracking.git"  # 例如："https://github.com/your_name/YOLOv8-DeepSORT-Object-Tracking.git"
PROJECT_DIR = Path("/content/YOLOv8-DeepSORT-Object-Tracking")

# 完整实验填 0；Colab 快速验证建议先用 300。
MAX_FRAMES = 0
DEVICE = "0"  # 有 GPU 时使用 "0"；没有 GPU 时改成 "cpu"

ROOT = Path.cwd()
if not (ROOT / "tools/course_design_full_eval.py").exists():
    ROOT = PROJECT_DIR

if not (ROOT / "tools/course_design_full_eval.py").exists():
    if REPO_URL.strip():
        if not PROJECT_DIR.exists():
            subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, str(PROJECT_DIR)], check=True)
        ROOT = PROJECT_DIR
    else:
        raise RuntimeError(
            "没有找到项目目录。请先把项目上传/复制到 /content/YOLOv8-DeepSORT-Object-Tracking，"
            "或者在本单元格填写 REPO_URL 后重新运行。"
        )

os.chdir(ROOT)

# 如果项目来自 Git，确保子模块也被拉取；已有目录时也会补齐缺失子模块。
# Colab 上有时会因为 safe.directory、网络抖动或子模块缓存返回 128，下面会打印 stderr 并尝试普通 clone 兜底。
SUBMODULE_FALLBACKS = {
    "third_party/ByteTrack": "https://github.com/FoundationVision/ByteTrack.git",
    "third_party/boxmot_eval": "https://github.com/mikel-brostrom/boxmot.git",
    "ultralytics/yolo/v8/detect/deep_sort_pytorch": "https://github.com/ZQPei/deep_sort_pytorch.git",
}

def run_git(cmd, check=False):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, check=False)
    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, result.stdout, result.stderr)
    return result

if (ROOT / ".git").exists() and (ROOT / ".gitmodules").exists():
    run_git(["git", "config", "--global", "--add", "safe.directory", str(ROOT)])
    run_git(["git", "submodule", "sync", "--recursive"])
    submodule_result = run_git(["git", "submodule", "update", "--init", "--recursive"])
    if submodule_result.returncode != 0:
        print("标准子模块更新失败，开始按 URL 补齐缺失子模块目录。")
        for rel_path, url in SUBMODULE_FALLBACKS.items():
            target = ROOT / rel_path
            if target.exists() and any(target.iterdir()):
                print("已存在，跳过:", target)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                target.rmdir()
            subprocess.run(["git", "clone", url, str(target)], check=True)
elif (ROOT / ".git").exists():
    print("未发现 .gitmodules，跳过子模块更新。")

VIDEO = ROOT / "video/video.mp4"
EVAL_SCRIPT = ROOT / "tools/course_design_full_eval.py"
ABLA_DIR = ROOT / "report_assets/ablation_colab"

print("Python:", sys.executable)
print("ROOT:", ROOT)
print("VIDEO:", VIDEO)
print("MAX_FRAMES:", MAX_FRAMES)
print("DEVICE:", DEVICE)

CalledProcessError: Command '['git', 'submodule', 'update', '--init', '--recursive']' returned non-zero exit status 128.

In [ ]:
# 2. 安装依赖
# 注意：不要在 Colab 使用本地 myenv 路径，这里统一使用当前 Colab Python。

def run(cmd, cwd=ROOT):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
run([sys.executable, "-m", "pip", "install", "lap", "easydict", "gdown"])

print("依赖安装完成")

$ /usr/bin/python3 -m pip install -r requirements.txt
$ /usr/bin/python3 -m pip install lap easydict gdown
依赖安装完成


In [ ]:
# 3. 检查 GPU、视频、DeepSORT 权重，并自动下载 YOLOv8n/m 权重

import torch

print("cuda_available =", torch.cuda.is_available())
print("gpu =", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

if not VIDEO.exists():
    raise FileNotFoundError(f"没有找到视频文件：{VIDEO}。请把你的场景视频放到 video/video.mp4，或修改 VIDEO 路径。")

DEEPSORT_CKPT = ROOT / "ultralytics/yolo/v8/detect/deep_sort_pytorch/deep_sort/deep/checkpoint/ckpt.t7"
if not DEEPSORT_CKPT.exists():
    raise FileNotFoundError(f"没有找到 DeepSORT 权重：{DEEPSORT_CKPT}。请把 ckpt.t7 放到该路径后再运行。")

WEIGHT_URLS = {
    "yolov8n.pt": "https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8n.pt",
    "yolov8m.pt": "https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8m.pt",
}

for name, url in WEIGHT_URLS.items():
    path = ROOT / name
    if path.exists():
        print("已存在:", path)
        continue
    print("下载:", name)
    urllib.request.urlretrieve(url, path)
    print("完成:", path)

print("检查完成")

cuda_available = True
gpu = Tesla T4


FileNotFoundError: 没有找到 DeepSORT 权重：/content/YOLOv8-DeepSORT-Object-Tracking/ultralytics/yolo/v8/detect/deep_sort_pytorch/deep_sort/deep/checkpoint/ckpt.t7。请把 ckpt.t7 放到该路径后再运行。

In [ ]:
# 4. 定义 4 组实验

EXPERIMENTS = [
    {
        "id": "E1",
        "name": "yolov8n_default",
        "model": "yolov8n.pt",
        "imgsz": 640,
        "conf": 0.25,
        "iou": 0.70,
        "output": ABLA_DIR / "yolov8n_default",
    },
    {
        "id": "E2",
        "name": "yolov8n_tuned",
        "model": "yolov8n.pt",
        "imgsz": 1280,
        "conf": 0.05,
        "iou": 0.85,
        "output": ABLA_DIR / "yolov8n_tuned",
    },
    {
        "id": "E3",
        "name": "yolov8m_default",
        "model": "yolov8m.pt",
        "imgsz": 640,
        "conf": 0.25,
        "iou": 0.70,
        "output": ABLA_DIR / "yolov8m_default",
    },
    {
        "id": "E4",
        "name": "yolov8m_tuned",
        "model": "yolov8m.pt",
        "imgsz": 1280,
        "conf": 0.05,
        "iou": 0.85,
        "output": ABLA_DIR / "yolov8m_tuned",
    },
]

for exp in EXPERIMENTS:
    print(exp["id"], exp["name"], exp["model"], "imgsz=", exp["imgsz"], "conf=", exp["conf"], "iou=", exp["iou"])

In [ ]:
# 5. 一键运行 4 组实验
# 输出目录：report_assets/ablation_colab/...

def build_command(exp):
    return [
        sys.executable,
        str(EVAL_SCRIPT),
        "--source", str(VIDEO),
        "--model", str(ROOT / exp["model"]),
        "--imgsz", str(exp["imgsz"]),
        "--conf", str(exp["conf"]),
        "--iou", str(exp["iou"]),
        "--device", DEVICE,
        "--max-frames", str(MAX_FRAMES),
        "--output", str(exp["output"]),
    ]

env = os.environ.copy()
env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

for exp in EXPERIMENTS:
    print("\n====", exp["id"], exp["name"], "====")
    cmd = build_command(exp)
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, env=env, check=True)

print("全部实验运行完成")

In [ ]:
# 6. 汇总结果表，并保存为 CSV/Markdown

import pandas as pd
from IPython.display import Markdown, display

def read_one_csv(path):
    if not path.exists():
        return None
    with path.open(newline="") as f:
        rows = list(csv.DictReader(f))
    return rows[0] if rows else None

def read_tracker_rows(path):
    if not path.exists():
        return []
    with path.open(newline="") as f:
        return list(csv.DictReader(f))

def fmt(row, key):
    if not row:
        return ""
    value = row.get(key, "")
    if value == "":
        return ""
    try:
        return f"{float(value):.3f}"
    except ValueError:
        return value

summary_rows = []
for exp in EXPERIMENTS:
    det = read_one_csv(exp["output"] / "detection_summary.csv")
    trackers = read_tracker_rows(exp["output"] / "tracker_summary.csv")
    tracker_by_name = {row.get("tracker"): row for row in trackers}
    deep = tracker_by_name.get("deepsort")
    byte = tracker_by_name.get("bytetrack_lite")
    summary_rows.append({
        "实验": exp["id"],
        "名称": exp["name"],
        "模型": exp["model"],
        "imgsz": exp["imgsz"],
        "conf": exp["conf"],
        "iou": exp["iou"],
        "平均检测数": fmt(det, "avg_detections"),
        "零检测帧率": fmt(det, "zero_detection_ratio"),
        "平均置信度": fmt(det, "avg_conf"),
        "小目标比例": fmt(det, "small_box_ratio"),
        "DeepSORT平均轨迹": fmt(deep, "avg_tracks"),
        "ByteTrack平均轨迹": fmt(byte, "avg_tracks"),
        "DeepSORT跳变均值": fmt(deep, "track_jump_mean"),
        "ByteTrack跳变均值": fmt(byte, "track_jump_mean"),
        "DeepSORT短轨迹比例": fmt(deep, "short_track_ratio"),
        "ByteTrack短轨迹比例": fmt(byte, "short_track_ratio"),
        "平均FPS": fmt(det, "avg_pipeline_fps"),
        "峰值显存MB": fmt(det, "peak_gpu_memory_mb"),
    })

df = pd.DataFrame(summary_rows)
ABLA_DIR.mkdir(parents=True, exist_ok=True)
csv_path = ABLA_DIR / "ablation_summary.csv"
md_path = ABLA_DIR / "ablation_summary.md"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
md_path.write_text(df.to_markdown(index=False) + "\n", encoding="utf-8")

display(df)
display(Markdown(md_path.read_text(encoding="utf-8")))
print("CSV:", csv_path)
print("Markdown:", md_path)

In [ ]:
# 7. 展示每组实验的关键图片

from IPython.display import Image, Markdown, display

image_names = [
    "counts_timeseries.png",
    "fps_timeseries.png",
    "latency_timeseries.png",
    "small_medium_large_boxes.png",
    "track_jump_comparison.png",
    "short_track_ratio.png",
]

for exp in EXPERIMENTS:
    display(Markdown(f"## {exp['id']} {exp['name']}"))
    for name in image_names:
        path = exp["output"] / name
        if path.exists():
            display(Markdown(f"**{name}**"))
            display(Image(filename=str(path)))
        else:
            print("missing:", path)

In [ ]:
# 8. 打包并下载全部结果

zip_base = Path("/content/yolov8_ablation_colab_results")
zip_path = shutil.make_archive(str(zip_base), "zip", str(ABLA_DIR))
print("压缩包:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print("自动下载不可用，可在 Colab 左侧文件栏手动下载:", zip_path)
    print(exc)